# Peter Testing Models on Restaurant Dataset

This restaurant health inspection dataset has appended Census indicators and Social Vulnerability Index indicators.


In [2]:
import os
import sys
import warnings

import numpy as np
import pandas as pd

if not sys.warnoptions:
    warnings.simplefilter("ignore")
    os.environ["PYTHONWARNINGS"] = "ignore"

In [3]:
# Load datasets
restaurants = pd.read_csv(
"/Users/pdeguz01/Documents/git/Data/IDS705_Final/Working Dataset/healthinspections_2024_withsvi_censustract")
restaurants

,INSPECTION_DATE,STORE_NAME,STREET_ADDRESS,CITY,ZIP5,SERVICE_DESCRIPTION,SCORE,GRADE,STREET_ADDRESS_LINE2,INSPDATE_YEAR,...,E_TWOMORE,E_OTHERRACE,EP_NOINT,EP_AFAM,EP_HISP,EP_ASIAN,EP_AIAN,EP_NHPI,EP_TWOMORE,EP_OTHERRACE
0,2024-01-02,3RD ST CHEVRON,3817 W 3RD ST,LOS ANGELES,90020,ROUTINE INSPECTION,90.0,A,NaN,2024,...,34.0,0.0,11.6,6.4,56.4,24.6,0.0,0.0,1.0,0.0
1,2024-01-02,7-ELEVEN #34473B,5905 SOUTH ST,LAKEWOOD,90713,ROUTINE INSPECTION,97.0,A,NaN,2024,...,222.0,19.0,4.1,6.6,28.6,23.5,0.3,1.4,4.9,0.4
2,2024-01-02,7-ELEVEN #39023B,4111 VENICE BLVD,LOS ANGELES,90019,ROUTINE INSPECTION,96.0,A,NaN,2024,...,39.0,22.0,9.3,24.3,51.6,10.8,0.6,0.0,1.1,0.6
3,2024-01-02,7-ELEVEN #39714,6051 HOLLYWOOD BLVD # 111,LOS ANGELES,90028,ROUTINE INSPECTION,94.0,A,NaN,2024,...,369.0,149.0,5.4,6.5,26.5,20.9,1.1,0.0,7.2,2.9
4,2024-01-02,AEIRLOOM-CATCHER,10550 RIVERSIDE DR,NORTH HOLLYWOOD,91602,ROUTINE INSPECTION,92.0,A,NaN,2024,...,111.0,26.0,7.9,2.0,21.7,5.3,0.0,0.0,4.7,1.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28685,2024-01-02,Whataburger #809,3400 W SLAUGHTER LN,AUSTIN,78748.0,Routine Inspection,94.0,NaN,NaN,2024,...,73.0,0.0,1.0,7.8,33.9,8.6,0.1,0.0,1.5,0.0
28686,2024-01-02,Nurturing Child Care Center,1054 SPRINGDALE RD,AUSTIN,78721.0,Routine Inspection,87.0,NaN,Bldg L,2024,...,48.0,0.0,8.2,2.2,57.6,1.9,0.0,0.0,1.0,0.0
28687,2024-01-02,Dry Creek Shell,3800 DRY CREEK DR,AUSTIN,78731.0,Routine Inspection,97.0,NaN,NaN,2024,...,133.0,0.0,6.9,2.5,15.4,4.9,0.0,0.0,2.3,0.0
28688,2024-01-02,OOB - PF - Pflugerville Donut,1616 FM 685,PFLUGERVILLE,78660.0,Routine Inspection,99.0,NaN,STE 104,2024,...,44.0,0.0,4.6,11.7,36.4,5.2,0.3,0.0,2.3,0.0


In [4]:
# Cleaning Service Description data
serv_des = [
    "ROUTINE INSPECTION",
    "REGULAR",
    "Routune Inspection",
    "Follow-Up Inspection",
    "FOLLOWUP",
]
# Keep only rows where SERVICE_DESCRIPTION is in the list
restaurants = restaurants[restaurants["SERVICE_DESCRIPTION"].isin(serv_des)]

# standardize SERVICE_DESCRIPTION values
standardization_map = {
    "ROUTINE INSPECTION": "Routine Inspection",
    "REGULAR": "Routine Inspection",
    "Routune Inspection": "Routine Inspection",
    "FOLLOW-UP INSPECTION": "Follow-Up Inspection",
    "FOLLOWUP": "Follow-Up Inspection",
}

restaurants["SERVICE_DESCRIPTION"] = restaurants["SERVICE_DESCRIPTION"].map(
    standardization_map
)

In [7]:
#Checking dataset for duplicates
duplicates = restaurants.duplicated(
    subset=["INSPECTION_DATE", "STORE_NAME", "STREET_ADDRESS", "CITY", "ZIP5"]
)
restaurants[duplicates]

print(
    "Number of duplicates:",
    len(restaurants[duplicates]),
    "\nPercentage of dataset that is duplicates: {:.2f}%".format(
        len(restaurants[duplicates]) / len(restaurants) * 100
    ),
)

Number of duplicates: 0 
Percentage of dataset that is duplicates: 0.00%


# Variable Cleaning and Manipulation

In [11]:
restaurants["SERVICE_DESCRIPTION"].value_counts()


SERVICE_DESCRIPTION
Routine Inspection      25381
Follow-Up Inspection       96
Name: count, dtype: int64

In [ ]:
# creating a dummy variable for service
#this creates two variables "SERVICE_Follow-Up Inspection" and "SERVICE_Routine Inspection"
restaurants["SERVICE_DESCRIPTION"].value_counts()
restaurants = pd.get_dummies(
    restaurants,
    columns=["SERVICE_DESCRIPTION"],
    prefix="SERVICE",
    drop_first=False,
)

#restaurants

,INSPECTION_DATE,STORE_NAME,STREET_ADDRESS,CITY,ZIP5,SCORE,GRADE,STREET_ADDRESS_LINE2,INSPDATE_YEAR,Latitude,...,EP_NOINT,EP_AFAM,EP_HISP,EP_ASIAN,EP_AIAN,EP_NHPI,EP_TWOMORE,EP_OTHERRACE,SERVICE_Follow-Up Inspection,SERVICE_Routine Inspection
0,2024-01-02,3RD ST CHEVRON,3817 W 3RD ST,LOS ANGELES,90020,90.0,A,NaN,2024,34.069180,...,11.6,6.4,56.4,24.6,0.0,0.0,1.0,0.0,False,True
1,2024-01-02,7-ELEVEN #34473B,5905 SOUTH ST,LAKEWOOD,90713,97.0,A,NaN,2024,33.859024,...,4.1,6.6,28.6,23.5,0.3,1.4,4.9,0.4,False,True
2,2024-01-02,7-ELEVEN #39023B,4111 VENICE BLVD,LOS ANGELES,90019,96.0,A,NaN,2024,34.044330,...,9.3,24.3,51.6,10.8,0.6,0.0,1.1,0.6,False,True
3,2024-01-02,7-ELEVEN #39714,6051 HOLLYWOOD BLVD # 111,LOS ANGELES,90028,94.0,A,NaN,2024,34.101820,...,5.4,6.5,26.5,20.9,1.1,0.0,7.2,2.9,False,True
4,2024-01-02,AEIRLOOM-CATCHER,10550 RIVERSIDE DR,NORTH HOLLYWOOD,91602,92.0,A,NaN,2024,34.152020,...,7.9,2.0,21.7,5.3,0.0,0.0,4.7,1.1,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26408,2024-11-13,OOB - PF - Dairy Queen,1701 W PECAN ST,PFLUGERVILLE,78660.0,100.0,NaN,NaN,2024,30.446837,...,7.9,3.1,41.1,14.6,0.0,1.5,1.7,0.5,False,False
27748,2024-07-30,Man Pasand Supermarket # 4,12625 N IH 35 NB SVRD,AUSTIN,78753.0,77.0,NaN,NaN,2024,30.409507,...,4.9,25.0,30.3,20.2,0.0,0.0,2.1,1.0,False,False
28116,2024-05-10,Placita's Restaurant,5310 S PLEASANT VALLEY RD,AUSTIN,78744.0,100.0,NaN,Bldg A,2024,30.194693,...,19.3,12.6,81.0,0.1,0.3,0.0,0.0,0.3,False,False
28356,2024-03-12,La Catedral Del Marisco #3,4118 S IH 35 SB SVRD,AUSTIN,78745.0,77.0,NaN,NaN,2024,30.215105,...,6.9,2.7,38.9,3.3,0.0,0.0,1.2,0.0,False,False


In [ ]:
#encoding datetime information and dropping the original column
restaurants["INSPECTION_DATE"] = restaurants["INSPECTION_DATE"].astype(
    "datetime64[ns]"
)
restaurants["INSPDATE_MONTH"] = restaurants["INSPECTION_DATE"].dt.month
restaurants["INSPDATE_DAY"] = restaurants["INSPECTION_DATE"].dt.day
restaurants = restaurants.drop(["INSPECTION_DATE"], axis=1)
restaurants

# Data Cleaning and Variable Deletion


In [ ]:
#checking NAN values
restaurants_check = restaurants.dropna()

# returning any rows with NAN values
assert (
    len(restaurants_check[restaurants_check.isna().any(axis=1)]) == 0
), "There are still rows with NAN values in the dataset"